# Visualize Model Predictions

This notebook loads a trained subgraph model and visualizes its predictions on a new sample.

NOFE predicts the high-resolution target grid directly. The six baselines (PCA, t-SNE, UMAP,
PHATE, SpatialPCA, Concord) are reduced on the low-resolution source features and then
inverse-distance-interpolated onto that same target grid, so every panel shows one value per
target node.

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


import napari
from scipy.interpolate import griddata
from pathlib import Path



project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
from plot_utilities import *
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

# Device setup
device = setup_device()
print(f'Using device: {device}')


from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import subprocess
import phate

# ---------------------------------------------------------------------------
# Method list + baseline backends.
#
# NOFE emits one value per *target* node. PCA/t-SNE/PHATE run in this process on the
# *source* features, so they need an extra step to reach the target grid: reduce the
# low-res source features, then inverse-distance-interpolate the embedding onto the
# target coordinates using the cross graph's edges. UMAP, Concord and SpatialPCA only
# exist in their own conda envs, so they run as subprocesses.
METHOD_NAMES = ['nofe', 'pca', 'tsne', 'umap', 'phate', 'spatialpca', 'concord']
BASELINE_NAMES = METHOD_NAMES[1:]

_spatialpca_proc = None
_spatialpca_dir = f'{project_root}/ERA5/spatialpca'
_spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'


def _spatialpca_server():
    """Start the persistent SpatialPCA R server once; return the process.

    Library load (SpatialPCA pulls in Seurat, SPARK, Rcpp) costs ~tens of seconds, so
    the server loads it once and then answers one request per stdin line.
    """
    global _spatialpca_proc
    if _spatialpca_proc is not None and _spatialpca_proc.poll() is None:
        return _spatialpca_proc
    _spatialpca_proc = subprocess.Popen(
        ['conda', 'run', '-n', 'spatialpca', '--no-capture-output',
         'Rscript', 'execute_spatialpca_server.R'],
        cwd=_spatialpca_dir, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
        stderr=subprocess.PIPE, text=True, bufsize=1,
        # R searches the *user* library (~/Library/R/<arch>/<ver>/library) before the
        # conda env's own library. CRAN-built packages there link the CRAN R framework
        # rather than conda's libR, and SpatialPCA's Rcpp .so then segfaults at dyn.load.
        env=dict(os.environ, R_LIBS_USER='/nonexistent'),
    )
    ready = _spatialpca_proc.stdout.readline().strip()
    if ready != 'READY':
        raise RuntimeError(
            f'spatialPCA server failed to start: {ready!r}\n'
            f'{_spatialpca_proc.stderr.read()}'
        )
    return _spatialpca_proc


def spatialpca_embed(X, coords, n_components=3):
    """SpatialPCA embedding of X (samples x features) located at coords (samples x d)."""
    proc = _spatialpca_server()
    np.savetxt(f'{_spatialpca_dir}/{_spatialpca_call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{_spatialpca_dir}/{_spatialpca_call_id}_coords.csv', coords, delimiter=',')

    proc.stdin.write(f'{_spatialpca_call_id} {n_components}\n')
    proc.stdin.flush()
    response = proc.stdout.readline().strip()

    os.remove(f'{_spatialpca_dir}/{_spatialpca_call_id}_X.csv')
    os.remove(f'{_spatialpca_dir}/{_spatialpca_call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {_spatialpca_call_id}: {response}')

    Y = np.loadtxt(f'{_spatialpca_dir}/res_{_spatialpca_call_id}.csv',
                   delimiter=',', skiprows=1)
    os.remove(f'{_spatialpca_dir}/res_{_spatialpca_call_id}.csv')
    return Y


def external_embed(env, workdir, script, buffer_id, X):
    """Run `script` under conda env `env`: it reads <buffer_id>.npy, writes res_<buffer_id>.npy."""
    np.save(f'{workdir}/{buffer_id}.npy', X)
    subprocess.run(f'cd {workdir} && conda run -n {env} python {script} {buffer_id}', shell=True)
    Y = np.load(f'{workdir}/res_{buffer_id}.npy')
    os.remove(f'{workdir}/{buffer_id}.npy')
    os.remove(f'{workdir}/res_{buffer_id}.npy')
    return Y


def embed_one(method, X, coords):
    """One baseline's 3-D embedding of source features X (samples x features)."""
    if method == 'pca':
        return PCA(n_components=3).fit_transform(X)
    if method == 'tsne':
        return TSNE(
            n_components=3, perplexity=30, learning_rate='auto', random_state=42
        ).fit_transform(X)
    if method == 'umap':
        return external_embed(
            'umap', f'{project_root}/ERA5/umap', 'execute_umap.py',
            f'subgraph_umap_buffer_{os.getpid()}', X,
        )
    if method == 'phate':
        return phate.PHATE(
            n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
        ).fit_transform(X)
    if method == 'spatialpca':
        return spatialpca_embed(X, coords)
    if method == 'concord':
        return external_embed(
            'concord', f'{project_root}/ERA5/concord', 'execute_concord.py',
            f'subgraph_concord_buffer_{os.getpid()}', X,
        )
    raise ValueError(f'unknown baseline {method!r}')


# t-SNE costs ~44 s at N_in=5000 and ~6 s at N_in=1000, and several panels read the same
# sample's embeddings, so each sample is embedded once and the result is reused.
# The cache holds a strong reference to the sample: cell 27 rebinds the global `sample`,
# which would otherwise let cell 12's object be collected and a later object reuse its
# id() -- a silent stale hit. Keeping the reference alive makes id() a safe key.
_source_embedding_cache = {}


def source_embeddings(sample):
    """{baseline: array(n_source, 3)} for every baseline, computed once per sample."""
    key = id(sample)
    if key not in _source_embedding_cache:
        X = sample.source.normed_values.detach().cpu().numpy()
        coords = sample.source.normed_coords.detach().cpu().numpy()
        _source_embedding_cache[key] = (sample, {
            method: embed_one(method, X, coords) for method in BASELINE_NAMES
        })
    return _source_embedding_cache[key][1]


def cached_sample_count():
    """Number of distinct samples embedded so far (used by the verification check)."""
    return len(_source_embedding_cache)


def interpolate_to_target(values, sample):
    """Inverse-distance-interpolate `values` (source nodes x F) onto the target nodes.

    Reuses Graph_Generator.inverse_distance_interpolation, the same routine the dataset
    builder uses to construct target node features. That routine runs torch ops on
    whatever device its inputs sit on, and the cross graph's coords/edges are on `device`
    (mps here) while sklearn/phate return CPU numpy, so all arguments are moved to CPU.
    """
    return Generator.inverse_distance_interpolation(
        normed_values_source=torch.as_tensor(
            np.asarray(values, dtype=np.float32), device='cpu'),
        normed_coords_source=sample.source.normed_coords.detach().cpu(),
        normed_coords_target=sample.target.normed_coords.detach().cpu(),
        cross_edge_index=sample.cross.edge_index.detach().cpu(),
    ).numpy()


def target_frame(sample, values):
    """lat/lon of the target grid + x1/x2/x3, the column set plot_utils.rgb consumes."""
    return pd.concat([
        pd.DataFrame({'lat': sample.target.meta.gps[:, 0],
                      'lon': sample.target.meta.gps[:, 1]}),
        pd.DataFrame(values, columns=['x1', 'x2', 'x3']),
    ], axis=1)


def source_frame(sample, values):
    """lat/lon of the source points + x1/x2/x3."""
    return pd.concat([
        pd.DataFrame({'lat': sample.source.meta.gps[:, 0],
                      'lon': sample.source.meta.gps[:, 1]}),
        pd.DataFrame(values, columns=['x1', 'x2', 'x3']),
    ], axis=1)


def predictions_for_sample(sample):
    """All 7 methods for one cross graph, every array with n_target rows.

    NOFE predicts the target nodes directly. Each baseline is reduced on the source
    features and then interpolated onto the target coordinates, so all seven arrays
    share one lat/lon table and are directly comparable.
    """
    preds = {'nofe': model(sample).detach().cpu().numpy()}
    for method, emb in source_embeddings(sample).items():
        preds[method] = interpolate_to_target(emb, sample)
    return preds


In [ ]:
# choose data
model_type = 'subgraph'


# ---------------------------------------------------------------------------
sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator


# Paths
trained_models_path = project_root / 'ERA5' / 'trained_models' / model_type
training_logs_path = trained_models_path / 'logs'
data_path = project_root.parent / 'ERA5' / 'data'  # Adjust if needed

print('Available models:\n')
files = sorted(os.listdir(trained_models_path), reverse=True)

for i, f in enumerate(files):
    if Path(f).suffix == ".pth":
        print(f'[{i}] {f}')



## 2. Load Model and Configuration

In [ ]:
# Set model ID
model_id = 'subgraph_tkernel_base2_d2_2026-04-25_t13-59-37' #'subgraph_tkernel_base_d3_2026-04-26_t12-02-03'  



# ---------------------------------------------------------------------------
if model_id.endswith('.pth'):
    model_id = model_id[:-4]

# Load full configuration (includes model, dataset, and training configs)
config_path = f'{training_logs_path}/configs/{model_id}.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# Load loss records
loss_df = pd.read_csv(f'{training_logs_path}/loss_records/{model_id}.csv')

print('Configuration:')
for key, value in config.items():
    print(f'  {key}: {value}')

print(f'\Training completed {len(loss_df)} epochs')
print(f'Final training loss: {loss_df["loss"].iloc[-1]:.4f}')
print(f'Final validation loss: {loss_df["val_loss"].iloc[-1]:.4f}')

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(loss_df['epoch'], loss_df['loss'], label='Training Loss', marker='o')
ax.plot(loss_df['epoch'], loss_df['val_loss'], label='Validation Loss', marker='s')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title(f'Training History - {model_id}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
data_folder_path = config["data_folder_path"]
dataset_save_path = config["dataset_save_path"]
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]

k_in = config["k_in"]
k_out = config["k_out"]
k_cross = config["k_cross"]
perplexity = config["perplexity"]
input_sampling_method = config["input_sampling_method"]
output_sampling_method = config["output_sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
interpolate_graph_features = config["interpolate_graph_features"]
vars = config["vars"]


In [ ]:
model = KernelNN_subgraph(
    width=config['width'],  # number features per node after lifting
    ker_width=config['ker_width'],  # dimension of single kernel layer
    depth=config['depth'],  # number of graph layers / message passings
    ker_in=config['edge_attr'],  # number of edge features
    known_node_dim=config['input_node_features'],  # only needed when lifting is applied within model
    unknown_node_dim=config['output_node_features'],
    out_width=config['out_width']
).to(device)



# Load trained weights
model_path = f'{trained_models_path}/{model_id}.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print(f'Model loaded: {model_id}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Create New Sample

Use the same functionality as dataset creation to generate a single sample.

In [ ]:
date = "2019-01-11"

bbox = lon_min, lat_min, lon_max, lat_max
Generator = Data_Generator(data_path)
Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')

In [ ]:
N_in = 400
input_sampling_method = "regular_subset"

in_graph = Generator.sample_graph(N=N_in, k=k_in, perplexity=perplexity, sampling_method=input_sampling_method, gen_feature_mode=feature_mode)
full_data = Generator.sample_graph(N=100000, k=0, perplexity=0, sampling_method='random', gen_feature_mode=feature_mode)

In [ ]:
N_out = 5000
output_sampling_method = "regular_subset"

out_graph = Generator.sample_graph(N=N_out, k=k_out, perplexity=perplexity, sampling_method=output_sampling_method, gen_feature_mode=feature_mode)
#out_graph = Generator.synthetic_graph(N=N_out, k=k_out, gen_feature_mode=feature_mode)
cross_graph = Generator.connect_graphs(in_graph=in_graph, out_graph=out_graph, k_cross=k_cross)
cross_graph["x"] = Generator.construct_output_node_features(cross_graph=cross_graph, interpolate=interpolate_graph_features, gen_feature_mode=feature_mode)
sample = cross_graph.to(device)

In [ ]:
figsize = (5, 5)

In [ ]:
print('predict results')
output = model(sample).detach().cpu().numpy()

In [ ]:
locations = pd.DataFrame({"lat": sample.target.meta.gps[:,0], "lon": sample.target.meta.gps[:,1]})

res_model = pd.DataFrame(
    {
        "x1": output[:, 0],
        "x2": output[:, 1],
        "x3": output[:, 2],
    }
)
res_model = pd.concat([locations, res_model], axis=1)

fig = rgb(res_model, figsize=figsize)

### PCA input points

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

# example input (replace with your array)
X = sample.source.x.detach().cpu().numpy() # shape: (samples, features)

# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)

locations = pd.DataFrame({"lat": sample.source.meta.gps[:,0], "lon": sample.source.meta.gps[:,1]})


# create pandas DataFrame
res_pca= pd.DataFrame(X_reduced, columns=["x1", "x2", "x3"])
res_pca = pd.concat([locations, res_pca], axis=1)

fig = rgb(res_pca, figsize=figsize)

## PCA output points

In [ ]:
from sklearn.decomposition import PCA

# example input (replace with your array)
X = sample.target.meta.x_true.detach().cpu().numpy() # shape: (samples, features)

# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)

locations_out = pd.DataFrame({"lat": sample.target.meta.gps[:,0], "lon": sample.target.meta.gps[:,1]})


# create pandas DataFrame
res_pca= pd.DataFrame(X_reduced, columns=["x1", "x2", "x3"])
res_pca = pd.concat([locations_out, res_pca], axis=1)

fig = rgb(res_pca, figsize=figsize)

## PCA all points

In [ ]:
from sklearn.decomposition import PCA

# example input (replace with your array)
X =full_data.x.detach().cpu().numpy() # shape: (samples, features)

# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)

locations_out = pd.DataFrame({"lat": full_data.meta.gps[:,0], "lon": full_data.meta.gps[:,1]})


# create pandas DataFrame
res_pca= pd.DataFrame(X_reduced, columns=["x1", "x2", "x3"])
res_pca = pd.concat([locations_out, res_pca], axis=1)

fig = rgb(res_pca, figsize=figsize)

# t-SNE outputs


In [ ]:

# example input (replace with your array)
X = sample.source.x.detach().cpu().numpy() # shape: (samples, features)

# apply PCA to reduce to 3 dimensions
pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)

locations_out = pd.DataFrame({"lat": sample.source.meta.gps[:,0], "lon": sample.source.meta.gps[:,1]})


# create pandas DataFrame
res_tsne= pd.DataFrame(pred_tsne, columns=["x1", "x2", "x3"])
res_tsne = pd.concat([locations_out, res_tsne], axis=1)

fig = rgb(res_tsne, figsize=figsize)

In [ ]:

# example input (replace with your array)
X = sample.target.meta.x_true.detach().cpu().numpy() # shape: (samples, features)

# apply PCA to reduce to 3 dimensions
pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)

locations_out = pd.DataFrame({"lat": sample.target.meta.gps[:,0], "lon": sample.target.meta.gps[:,1]})


# create pandas DataFrame
res_tsne= pd.DataFrame(pred_tsne, columns=["x1", "x2", "x3"])
res_tsne = pd.concat([locations_out, res_tsne], axis=1)

fig = rgb(res_tsne, figsize=figsize)

## PHATE (target resolution)

In [ ]:
preds = predictions_for_sample(sample)
fig = rgb(target_frame(sample, preds['phate']), figsize=figsize)

## SpatialPCA (target resolution)

In [ ]:
preds = predictions_for_sample(sample)
fig = rgb(target_frame(sample, preds['spatialpca']), figsize=figsize)

## Concord (target resolution)

In [ ]:
preds = predictions_for_sample(sample)
fig = rgb(target_frame(sample, preds['concord']), figsize=figsize)

# compare methods for different resolutions

In [ ]:
date = "2019-01-11"

bbox = lon_min, lat_min, lon_max, lat_max
Generator = Data_Generator(data_path)
Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')

input_sampling_method = "regular_subset"

N_out = 5000
output_sampling_method = "regular_subset"
figsize = (5, 5)

data = []

for N_in in [200, 400, 1000, 5000]:
    in_graph = Generator.sample_graph(N=N_in, k=k_in, perplexity=perplexity, sampling_method=input_sampling_method, gen_feature_mode=feature_mode)
    out_graph = Generator.sample_graph(N=N_out, k=k_out, perplexity=perplexity, sampling_method=output_sampling_method, gen_feature_mode=feature_mode)
    #out_graph = Generator.synthetic_graph(N=N_out, k=k_out, gen_feature_mode=feature_mode)
    cross_graph = Generator.connect_graphs(in_graph=in_graph, out_graph=out_graph, k_cross=k_cross)
    cross_graph["x"] = Generator.construct_output_node_features(cross_graph=cross_graph, interpolate=interpolate_graph_features, gen_feature_mode=feature_mode)
    sample = cross_graph.to(device)
    
    data.append(sample)

In [ ]:
for sample in data:
    preds = predictions_for_sample(sample)

    fig = rgb(target_frame(sample, preds['nofe']), figsize=figsize)

# PCA

In [ ]:
for sample in data:
    # source resolution: the reduction evaluated only where it was fitted
    fig_source = rgb(
        source_frame(sample, source_embeddings(sample)['pca']), figsize=figsize)

    # target resolution: the same embedding interpolated onto the output grid
    preds = predictions_for_sample(sample)
    fig_target = rgb(target_frame(sample, preds['pca']), figsize=figsize)

# t-SNE

In [ ]:
for sample in data:
    # source resolution: the reduction evaluated only where it was fitted
    fig_source = rgb(
        source_frame(sample, source_embeddings(sample)['tsne']), figsize=figsize)

    # target resolution: the same embedding interpolated onto the output grid
    preds = predictions_for_sample(sample)
    fig_target = rgb(target_frame(sample, preds['tsne']), figsize=figsize)

In [ ]:
for sample in data:
    # source resolution: the reduction evaluated only where it was fitted
    fig_source = rgb(
        source_frame(sample, source_embeddings(sample)['umap']), figsize=figsize)

    # target resolution: the same embedding interpolated onto the output grid
    preds = predictions_for_sample(sample)
    fig_target = rgb(target_frame(sample, preds['umap']), figsize=figsize)

# PHATE

In [ ]:
for sample in data:
    preds = predictions_for_sample(sample)

    fig_target = rgb(target_frame(sample, preds['phate']), figsize=figsize)

# SpatialPCA

In [ ]:
for sample in data:
    preds = predictions_for_sample(sample)

    fig_target = rgb(target_frame(sample, preds['spatialpca']), figsize=figsize)

# Concord

In [ ]:
for sample in data:
    preds = predictions_for_sample(sample)

    fig_target = rgb(target_frame(sample, preds['concord']), figsize=figsize)